In [ ]:
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

# 1. 模拟数据：3个序列，长度分别是 3, 5, 1 (维度 dim=2)
# 也就是：长文章、中文章、短文章
seq1 = torch.tensor([[2.1, 2.1], [2.2, 2.2], [2.3, 2.3]], dtype=torch.float)
seq2 = torch.tensor(
    [[1.1, 1.1], [1.2, 1.2], [1.3, 1.3], [1.4, 1.4], [1.5, 1.5]], dtype=torch.float
)
seq3 = torch.tensor([[3.1, 3.1]], dtype=torch.float)

sequences = [seq1, seq2, seq3]
# 必须记录原始长度
lengths = torch.tensor([len(s) for s in sequences])

print(f"原始长度: {lengths}")

# 2. Pad: 变成整齐的矩阵
# batch_first=True 让 batch 维度在最前面 (Batch, Seq_Len, Feature)
padded_seqs = pad_sequence(sequences, batch_first=True, padding_value=0)
print(f"\nPad 后的形状: {padded_seqs.shape}")
# 预期: [3, 5, 2] -> [Batch Size, Max Len, Feature Dim]
print(padded_seqs)

原始长度: tensor([3, 5, 1])

Pad 后的形状: torch.Size([3, 5, 2])
tensor([[[2.1000, 2.1000],
         [2.2000, 2.2000],
         [2.3000, 2.3000],
         [0.0000, 0.0000],
         [0.0000, 0.0000]],

        [[1.1000, 1.1000],
         [1.2000, 1.2000],
         [1.3000, 1.3000],
         [1.4000, 1.4000],
         [1.5000, 1.5000]],

        [[3.1000, 3.1000],
         [0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.0000, 0.0000],
         [0.0000, 0.0000]]])


In [35]:
# 3. Pack: 压缩
# enforce_sorted=False 允许我们不手动按长度降序排列
packed_seqs = pack_padded_sequence(
    padded_seqs, lengths, batch_first=True, enforce_sorted=False
)

print(f"\nPack 后的数据结构:\n {packed_seqs}")
# 主要由 data 和 batch_sizes 组合而成
# data: 把所有有效数据按照 seq_len 拼在了一起, 使用的时候按照 batch_size 取特定的数量
# batch_sizes: [3, 2, 2, 1, 1] -> 表示每个时间步有多少个有效数据


Pack 后的数据结构:
 PackedSequence(data=tensor([[1.1000, 1.1000],
        [2.1000, 2.1000],
        [3.1000, 3.1000],
        [1.2000, 1.2000],
        [2.2000, 2.2000],
        [1.3000, 1.3000],
        [2.3000, 2.3000],
        [1.4000, 1.4000],
        [1.5000, 1.5000]]), batch_sizes=tensor([3, 2, 2, 1, 1]), sorted_indices=tensor([1, 0, 2]), unsorted_indices=tensor([1, 0, 2]))


In [ ]:
# 4. 模拟 LSTM 处理
lstm = nn.LSTM(input_size=2, hidden_size=4, batch_first=True)
# LSTM 可以直接接收 PackedSequence 对象！输出也是 PackedSequence 对象！
packed_output, (hn, cn) = lstm(packed_seqs)
print(f"LSTM 输出的形状为: {packed_output.data.shape}")
# batch_size 和 seq_len 都不再存在,变成了: [sum_seq_len, hidden_dim]
# 后续为了进一步进入其他层,需要还原到: [batch_size, seq_len, hidden_dim]
print(f"隐藏层输出的形状为: {hn.data.shape}")

LSTM 输出的形状为: torch.Size([9, 4])
输出的形状为: torch.Size([1, 3, 4])


In [ ]:
packed_output

PackedSequence(data=tensor([[ 0.0725,  0.2471, -0.0169,  0.0585],
        [ 0.0530,  0.3660, -0.0070,  0.0909],
        [ 0.0362,  0.4111, -0.0026,  0.0641],
        [ 0.0958,  0.3382, -0.0214,  0.0918],
        [ 0.0609,  0.4530, -0.0090,  0.1034],
        [ 0.0990,  0.3809, -0.0225,  0.1068],
        [ 0.0589,  0.4820, -0.0095,  0.1022],
        [ 0.0960,  0.4060, -0.0220,  0.1142],
        [ 0.0914,  0.4237, -0.0210,  0.1182]], grad_fn=<CatBackward0>), batch_sizes=tensor([3, 2, 2, 1, 1]), sorted_indices=tensor([1, 0, 2]), unsorted_indices=tensor([1, 0, 2]))

In [ ]:
# 5. Unpack: 还原
output, output_lens = pad_packed_sequence(packed_output, batch_first=True)

print(f"\n还原后的输出形状: {output.shape}")
# 预期: [3, 5, 4] -> [Batch Size, Max Len, Hidden Dim]


还原后的输出形状: torch.Size([3, 5, 4])


In [ ]:
output_lens  # 每个句子有效 seq_len 数量

tensor([3, 5, 1])

In [30]:
# 注意：还原后的 output 中，原本 padding 的位置输出也是 0
output

tensor([[[ 0.0124,  0.0075, -0.0462, -0.0425],
         [ 0.0200,  0.0266, -0.0525, -0.0637],
         [ 0.0245,  0.0446, -0.0499, -0.0751],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000]],

        [[-0.0062, -0.0267, -0.0733,  0.0622],
         [-0.0083, -0.0368, -0.0937,  0.0677],
         [-0.0071, -0.0379, -0.0994,  0.0563],
         [-0.0040, -0.0338, -0.0996,  0.0405],
         [ 0.0001, -0.0267, -0.0971,  0.0245]],

        [[ 0.0168,  0.0594, -0.0154, -0.1212],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000]]],
       grad_fn=<IndexSelectBackward0>)

In [6]:
from torch.nn.utils.rnn import pad_sequence

a = torch.ones(5, 8)
b = torch.ones(2, 8)
c = torch.ones(1, 8)
pad_result = pad_sequence([a, b, c], batch_first=True)
print(pad_result.shape)
print(pad_result)

torch.Size([3, 5, 8])
tensor([[[1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.]],

        [[1., 1., 1., 1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.]],

        [[1., 1., 1., 1., 1., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.]]])


In [18]:
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

seq = torch.tensor([[1, 2, 0], [3, 0, 0], [4, 5, 6]])
lens = [2, 1, 3]
packed = pack_padded_sequence(seq, lens, batch_first=True, enforce_sorted=False)
packed


PackedSequence(data=tensor([4, 1, 3, 5, 2, 6]), batch_sizes=tensor([3, 2, 1]), sorted_indices=tensor([2, 0, 1]), unsorted_indices=tensor([1, 2, 0]))

sorted_indices 记录了：原始数据的每一项，现在跑到了排序后序列的哪个位置，其中索引以**原数据为标准**。

unsorted_indices 记录了：排序后的每一项，原本是在原始序列的哪个位置，其中索引以**排序后数据为标准**。

以上述例子说明：排序后的数据为 [3, 2, 1], 对应原始索引(sorted_indices)为: [2, 0, 1]; 如果要对应回原来顺序,排列应该是: [1, 2, 0] (3 的位置原来在最后)

In [19]:
seq_unpacked, lens_unpacked = pad_packed_sequence(packed, batch_first=True)
seq_unpacked, lens_unpacked

(tensor([[1, 2, 0],
         [3, 0, 0],
         [4, 5, 6]]),
 tensor([2, 1, 3]))

In [33]:
from torch.nn.utils.rnn import pack_sequence

# 1. 模拟数据：3个序列，长度分别是 5, 3, 1 (维度 dim=2)
# 也就是：长文章、中文章、短文章
seq1 = torch.tensor(
    [[1.1, 1.1], [1.2, 1.2], [1.3, 1.3], [1.4, 1.4], [1.5, 1.5]], dtype=torch.float
)
seq2 = torch.tensor([[2.1, 2.1], [2.2, 2.2], [2.3, 2.3]], dtype=torch.float)
seq3 = torch.tensor([[3.1, 3.1]], dtype=torch.float)
seqs = [seq2, seq1, seq3]

# 直接 Pack！不需要 Pad，也不需要手动传 lengths
# 注意：默认 enforce_sorted=True，如果数据乱序设为 False
packed = pack_sequence(seqs, enforce_sorted=False)
packed

PackedSequence(data=tensor([[1.1000, 1.1000],
        [2.1000, 2.1000],
        [3.1000, 3.1000],
        [1.2000, 1.2000],
        [2.2000, 2.2000],
        [1.3000, 1.3000],
        [2.3000, 2.3000],
        [1.4000, 1.4000],
        [1.5000, 1.5000]]), batch_sizes=tensor([3, 2, 2, 1, 1]), sorted_indices=tensor([1, 0, 2]), unsorted_indices=tensor([1, 0, 2]))